In [1]:
import findspark
findspark.init()
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import re


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder. \
    appName("pyspark-1"). \
    getOrCreate()

### Read data

In [3]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)

# Replace space in column names
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.replace(" ", "_"))
    
df.printSchema()
df.show(5) 

root
 |-- Job_ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting_Type: string (nullable = true)
 |-- #_Of_Positions: string (nullable = true)
 |-- Business_Title: string (nullable = true)
 |-- Civil_Service_Title: string (nullable = true)
 |-- Title_Code_No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job_Category: string (nullable = true)
 |-- Full-Time/Part-Time_indicator: string (nullable = true)
 |-- Salary_Range_From: string (nullable = true)
 |-- Salary_Range_To: string (nullable = true)
 |-- Salary_Frequency: string (nullable = true)
 |-- Work_Location: string (nullable = true)
 |-- Division/Work_Unit: string (nullable = true)
 |-- Job_Description: string (nullable = true)
 |-- Minimum_Qual_Requirements: string (nullable = true)
 |-- Preferred_Skills: string (nullable = true)
 |-- Additional_Information: string (nullable = true)
 |-- To_Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work_Locatio

### Data Profiling

In [4]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 2946
Total Columns: 28


In [5]:
# Null count in each column
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

+------+------+------------+--------------+--------------+-------------------+-------------+-----+------------+-----------------------------+-----------------+---------------+----------------+-------------+------------------+---------------+-------------------------+----------------+----------------------+--------+-----------+---------------+-------------------+---------------------+------------+----------+---------------+------------+
|Job_ID|Agency|Posting_Type|#_Of_Positions|Business_Title|Civil_Service_Title|Title_Code_No|Level|Job_Category|Full-Time/Part-Time_indicator|Salary_Range_From|Salary_Range_To|Salary_Frequency|Work_Location|Division/Work_Unit|Job_Description|Minimum_Qual_Requirements|Preferred_Skills|Additional_Information|To_Apply|Hours/Shift|Work_Location_1|Recruitment_Contact|Residency_Requirement|Posting_Date|Post_Until|Posting_Updated|Process_Date|
+------+------+------------+--------------+--------------+-------------------+-------------+-----+------------+---------

### Data Cleaning

In [6]:
# Removes $ and , characters. Converts the cleaned value into a double
df = (
    df.withColumn(
        "salary_from",
        regexp_replace(col("Salary_Range_From"), "[$,]", "").cast("double")
    )
    .withColumn(
        "salary_to",
        regexp_replace(col("Salary_Range_To"), "[$,]", "").cast("double")
    )
)

# Calculate average salary
df = df.withColumn(
    "avg_salary",
    (col("salary_from") + col("salary_to")) / 2
)

df = df.filter(col("avg_salary").isNotNull())

### Feature Engineering

In [7]:
# Calculate salary_range_diff (salary_to - salary_from)
df = df.withColumn(
    "salary_range_diff",
    col("salary_to") - col("salary_from")
)

# Convert Posting_Date to Date type
df = df.withColumn(
    "Posting_Date",
    to_timestamp(col("Posting_Date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
)

df = df.withColumn("posting_year", year(col("Posting_Date")))

df = df.withColumn(
    "Posting_Date",
    date_format(col("Posting_Date"), "MM/dd/yyyy")
)

# Calculate age of the job
df = df.withColumn(
    "job_posting_age_days",
    datediff(current_date(), col("Posting_Date"))
)

# Add "has_masters_degree" Flag
df = df.withColumn(
    "has_masters_degree",
    when(lower(col("Minimum_Qual_Requirements")).contains("master"), 1).otherwise(0)
)



### KPI

In [8]:
# number of jobs posting per category (Top 10)
top_categories = (
    df.groupBy("Job_Category")
      .count()
      .orderBy(desc("count"))
      .limit(10)
)
top_categories.show()


+--------------------+-----+
|        Job_Category|count|
+--------------------+-----+
|Engineering, Arch...|  504|
|Technology, Data ...|  313|
|       Legal Affairs|  226|
|Public Safety, In...|  182|
|Building Operatio...|  181|
|Finance, Accounti...|  169|
|Administration & ...|  134|
|Constituent Servi...|  129|
|              Health|  125|
|Policy, Research ...|  124|
+--------------------+-----+



In [9]:
# salary distribution per job category
salary_distribution = (
    df.groupBy("Job_Category")
      .agg(avg("avg_salary").alias("category_avg_salary"))
      .orderBy(desc("category_avg_salary"))
)

salary_distribution.show()


+--------------------+-------------------+
|        Job_Category|category_avg_salary|
+--------------------+-------------------+
|Administration & ...|           218587.0|
|Engineering, Arch...|           198518.0|
|Engineering, Arch...|           196042.5|
|Health Policy, Re...|           128694.5|
|Engineering, Arch...|           128247.5|
|Engineering, Arch...|           128247.5|
|Communications & ...|           125000.0|
|Administration & ...|           118287.0|
|Constituent Servi...| 116900.33333333333|
|Constituent Servi...|           103680.5|
|Technology, Data ...|           96373.45|
|Administration & ...|            95000.0|
|Administration & ...|            92500.0|
|Administration & ...|          91497.575|
|Administration & ...|  91174.22222222222|
|Administration & ...|            91000.0|
|Administration & ...|            90000.0|
|Engineering, Arch...|  89857.43548387097|
|Engineering, Arch...|            89730.0|
|Legal Affairs Pol...|  88898.66666666667|
+----------

In [9]:
# correlation between the higher degree and the salary
correlation = df.stat.corr("has_masters_degree", "avg_salary")
print("Correlation between Masters Degree & Salary:", correlation)

Correlation between Masters Degree & Salary: 0.21111185524557488


In [10]:
# job posting having the highest salary per agency
windowSpec = Window.partitionBy("Agency").orderBy(desc("avg_salary"))

highest_salary_per_agency = (
    df.withColumn("rank", row_number().over(windowSpec))
      .filter(col("rank") == 1)
      .select("Agency", "Business_Title", "avg_salary")
)

highest_salary_per_agency.show()

+--------------------+--------------------+----------+
|              Agency|      Business_Title|avg_salary|
+--------------------+--------------------+----------+
|LANDMARKS PRESERV...|LANDMARKS PRESERV...|   60103.5|
|OFFICE OF COLLECT...|COLLEGE AIDE - CL...|     9.555|
|     FIRE DEPARTMENT|Senior Enterprise...|  120474.5|
|ADMIN FOR CHILDRE...|Assistant Commiss...|  117474.5|
|MANHATTAN COMMUNI...| Community Assistant|      19.0|
|      TAX COMMISSION|       CITY ASSESSOR|   45088.5|
|HRA/DEPT OF SOCIA...|EXECUTIVE DIRECTO...|  112527.5|
|TAXI & LIMOUSINE ...|Executive Directo...|  140000.0|
|EQUAL EMPLOY PRAC...|Director of Learn...|   67970.0|
|DEPARTMENT OF BUS...|EXECUTIVE DIRECTO...|  111377.0|
|DEPT OF DESIGN & ...|Associate Commiss...|  151795.0|
|TEACHERS RETIREME...|AGENCY ATTORNEY I...|   69078.5|
|DEPARTMENT OF COR...|Assistant Commiss...|  137500.0|
|FINANCIAL INFO SV...|Windows Technical...|  135000.0|
|OFFICE OF EMERGEN...|DISABILITY ACCESS...|   70995.0|
|HOUSING P

In [11]:
# job positings average salary per agency for the last 2 years
last_2_years = df.filter(
    col("posting_year") >= year(current_date()) - 9
)

avg_salary_last_2_years = (
    last_2_years.groupBy("Agency")
        .agg(avg("avg_salary").alias("agency_avg_salary"))
        .orderBy(desc("agency_avg_salary"))
)

avg_salary_last_2_years.show()

+--------------------+-----------------+
|              Agency|agency_avg_salary|
+--------------------+-----------------+
|CONFLICTS OF INTE...|         135000.0|
|BUSINESS INTEGRIT...|          95000.0|
|FINANCIAL INFO SV...|          93994.5|
|MAYORS OFFICE OF ...|87357.14285714286|
|    CONSUMER AFFAIRS|87110.28571428571|
|OFFICE OF THE COM...|85235.71428571429|
|HOUSING PRESERVAT...|84882.17857142857|
|DEPT OF INFO TECH...|83375.72700000001|
|PRESIDENT BOROUGH...|          82500.0|
|DEPT OF YOUTH & C...|        80697.875|
|      LAW DEPARTMENT|80657.18897230769|
|NYC HOUSING AUTHO...|78538.13848484849|
|HUMAN RIGHTS COMM...|   78094.03029375|
|DEPT OF DESIGN & ...|     77807.171875|
|NYC EMPLOYEES RET...|74775.33333333333|
|DISTRICT ATTORNEY...|         70691.25|
|DISTRICT ATTORNEY...|70330.92857142857|
|OFFICE OF MANAGEM...|69765.86896551723|
|TEACHERS RETIREME...|          69078.5|
|DEPARTMENT OF PRO...|68229.43111111112|
+--------------------+-----------------+
only showing top

In [12]:
# highest paid skills in the US market
split_pattern = r"(?:•|\-{1,2}|\d+\.\s+|;\s*|\n)"

df_exploded = df.withColumn(
    "skill_item",
    explode(split(col("Preferred_Skills"), split_pattern))   # ← fixed column name
).withColumn(
    "skill_item",
    trim(col("skill_item"))
).filter(
    col("skill_item").isNotNull() &
    (length(col("skill_item")) > 3)
)


df_exploded = df_exploded.withColumn("skill_item", lower(col("skill_item")))


top_skills = (
    df_exploded
    .filter(col("skill_item") != "Other") 
    .groupBy("skill_item")
    .agg(avg("avg_salary").alias("skill_avg_salary"))
    .orderBy(desc("skill_avg_salary"))
    .limit(10)
)


### Store Processed Data 

In [13]:
# Drop unnecessary columns. Store Processed data
DROP_COLS = [
    'Job_Description', 'To_Apply', 'Minimum_Qual_Requirements',
    'Additional_Information', 'Recruitment_Contact', 'Title_Code_No'
]

to_drop = [c for c in DROP_COLS if c in df.columns]

print(f"Removing {len(to_drop)} columns: {to_drop}")
df = df.drop(*to_drop)

df.coalesce(1).write.mode("overwrite").parquet("/dataset/processed/nyc_jobs_processed_parquet")
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/dataset/processed/nyc_jobs_processed_CSV")


Removing 6 columns: ['Job_Description', 'To_Apply', 'Minimum_Qual_Requirements', 'Additional_Information', 'Recruitment_Contact', 'Title_Code_No']


### Test Cases

In [14]:
def test_avg_salary_not_null():
    assert df.filter(col("avg_salary").isNull()).count() == 0, \
        "avg_salary contains null values"

test_avg_salary_not_null()
print("Test 1 Passed")

def test_salary_range_positive():
    assert df.filter(col("salary_range_diff") < 0).count() == 0, \
        "Found negative salary range"

test_salary_range_positive()
print("Test 2 Passed")

def test_masters_flag_binary():
    invalid = df.filter(~col("has_masters_degree").isin(0,1)).count()
    assert invalid == 0, "Invalid values found in has_masters_degree"

test_masters_flag_binary()
print("Test 3 Passed")


Test 1 Passed
Test 2 Passed
Test 3 Passed


### Sample function

In [48]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result